In [ ]:
import numpy as np


tokens = ["I", "love", "AI"]
seq_len = len(tokens)
d_model = 4

np.random.seed(0)

# Token embeddings (pretend they came from an embedding layer)
embeddings = np.random.rand(seq_len, d_model)

print("Tokens:", tokens)
print("\nEmbeddings:\n", embeddings)


Tokens: ['I', 'love', 'AI']

Embeddings:
 [[0.5488135  0.71518937 0.60276338 0.54488318]
 [0.4236548  0.64589411 0.43758721 0.891773  ]
 [0.96366276 0.38344152 0.79172504 0.52889492]]


In [ ]:
# Similarity scores between all token pairs
# Cosine similarity is conceptually helpful, but computationally replaced by dot products in Transformers.
similarity_scores = embeddings @ embeddings.T 

scaled_scores = similarity_scores / np.sqrt(d_model)

print("\nSimilarity Scores:\n", similarity_scores)
print("\nScaled Scores:\n", scaled_scores)



Similarity Scores:
 [[1.47291346 1.44411773 1.56851324]
 [1.44411773 1.58340425 1.47402593]
 [1.56851324 1.47402593 1.98223169]]

Scaled Scores:
 [[0.73645673 0.72205887 0.78425662]
 [0.72205887 0.79170212 0.73701297]
 [0.78425662 0.73701297 0.99111584]]


In [4]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

attention_weights = softmax(scaled_scores)

print("\nAttention Weights:\n", attention_weights)



Attention Weights:
 [[0.32952555 0.32481508 0.34565936]
 [0.32391894 0.3472818  0.32879926]
 [0.31410346 0.29960914 0.3862874 ]]


In [ ]:
# Context Vectors

context_vectors = attention_weights @ embeddings

print("\nContext Vectors:\n", context_vectors)



Context Vectors:
 [[0.6515566  0.57800947 0.61442803 0.65203173]
 [0.64175029 0.58204594 0.60753115 0.66009477]
 [0.67156585 0.56627786 0.6262686  0.64263848]]


#### Large Configuration

In [7]:
np.random.seed(42)

# ----------------------------
# Large-scale configuration
# ----------------------------
batch_size = 8        # e.g., 8 sentences
seq_len = 128         # long sequence
d_model = 256         # realistic embedding size

# Simulated embeddings
# Shape: (batch_size, seq_len, d_model)
embeddings = np.random.randn(batch_size, seq_len, d_model)

print("Embeddings:", embeddings)
print("==========================")
print("Embeddings shape:", embeddings.shape)

Embeddings: [[[ 0.49671415 -0.1382643   0.64768854 ...  1.03246526 -1.51936997
   -0.48423407]
  [ 1.26691115 -0.70766947  0.44381943 ... -0.83095012  0.27045683
   -0.05023811]
  [-0.23894805 -0.90756366 -0.57677133 ... -0.70317643 -0.03498849
    1.77080064]
  ...
  [ 0.73201427 -1.61156562 -0.67634599 ...  0.02029628  1.33841508
   -0.47352563]
  [-1.7325688   0.4405669  -0.75559825 ... -1.41106761  0.24610642
    1.11636145]
  [ 0.34801752  0.45783305  0.1679589  ... -0.27404025 -1.02916233
    0.11805232]]

 [[-1.07411893 -0.52663344  0.7011298  ... -2.10228537  0.7641678
    1.64599856]
  [ 1.54274879 -1.40808925  1.01635683 ...  1.2067352   0.92179941
    0.78918574]
  [ 2.30151768  1.07661759 -0.849317   ... -0.49528095  1.03401125
    0.24441276]
  ...
  [-1.31841007 -0.30088289 -0.30432466 ... -0.43489262 -0.51101035
    1.22096185]
  [-0.21424977 -0.37309086  1.12880514 ...  0.61536325 -1.94165054
    0.83796642]
  [ 0.06491573 -0.03287519  1.19042966 ...  1.41374069 -0.6055

In [14]:
# S =E ⋅ ET

# Similarity scores
# Shape: (batch_size, seq_len, seq_len)

# -- For each sentence

# -- Every token compares itself with every other token

# -- 128 × 128 attention map per sentence

similarity_scores = np.matmul(
    embeddings,
    embeddings.transpose(0, 2, 1)
)

scaled_scores = similarity_scores / np.sqrt(d_model)

print("Similarity scores shape:", similarity_scores.shape)


Similarity scores shape: (8, 128, 128)


In [15]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

attention_weights = softmax(scaled_scores, axis=-1)

print("Attention weights shape:", attention_weights.shape)


Attention weights shape: (8, 128, 128)


In [ ]:
# The Context Vector

'''

tokens = ["I", "love", "AI"]

E = np.array([
    [1.0, 0.0],   # "I"
    [0.8, 0.6],   # "love"
    [0.0, 1.0]    # "AI"
])

# S = E @ E.T

S =
[[1.0  0.8  0.0]
 [0.8  1.0  0.6]
 [0.0  0.6  1.0]]

# A = softmax(similarity score)

A ≈
[[0.46  0.38  0.16]   # I attends mostly to I and love
 [0.30  0.37  0.33]   # love attends to everyone
 [0.16  0.38  0.46]]  # AI attends to AI and love
 
# Ci = A @ E

The context vector is what the word means to sentence.

context_I = 0.46 * [1.0, 0.0] + 0.38 * [0.8, 0.6] + 0.16 * [0.0, 1.0]

= [0.46 + 0.30 + 0.00, 0.00 + 0.23 + 0.16]

≈ [0.76, 0.39]

'''

In [18]:
# Context = softmax((E · Eᵀ) / √d) · E

context_vectors = np.matmul(attention_weights, embeddings)

print("Context vectors shape:", context_vectors.shape)


Context vectors shape: (8, 128, 256)
